# Chapter 20 — Binary Outcomes

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch20_binary_outcomes.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 20.1 First, reproduce the published number

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

from indiadata import read_plfs, labour_status, labour_rates

HH_KEYS = ["qtr", "visit", "sec", "st", "mfsu", "seg", "sss", "ssu"]
p = read_plfs("perv1", columns=["seg", "sss", "ssu", "sex", "age", "marst",
                                "gedu_lvl", "pas", "sas"])
p["status"] = labour_status(p["pas"], p["sas"])

women15 = p[(p["sex"] == 2) & (p["age"] >= 15)]
ours = women15.groupby("sec").apply(labour_rates)["LFPR"]
ours["all"] = labour_rates(women15)["LFPR"]
check = pd.DataFrame({"ours": ours.round(1),
                      "MoSPI": [47.6, 28.0, 41.7]}).rename(
    index={1: "rural", 2: "urban"})
check["match"] = check["ours"] == check["MoSPI"]
check

## 20.2 Merging persons and households

In [ ]:
hh = [p[k] for k in HH_KEYS]
p["kids05"] = (p["age"] <= 5).groupby(hh, observed=True).transform("sum")
p["elderly"] = (p["age"] >= 60).groupby(hh, observed=True).transform("sum")
p["members"] = p["age"].groupby(hh, observed=True).transform("size")

In [ ]:
h = read_plfs("hhv1", columns=["seg", "sss", "ssu", "hh_size", "relg", "sg",
                               "hce_tot"])
h = h[HH_KEYS + ["hh_size", "relg", "sg", "hce_tot"]]

w = p[(p["sex"] == 2) & p["age"].between(25, 59)].merge(
    h, on=HH_KEYS, how="left", validate="m:1", indicator=True)
del p, h, women15                                    # free memory
print(w["_merge"].value_counts().to_string())
print(f"Household size in hhv1 equals the number of person records: "
      f"{(w['hh_size'] == w['members']).mean():.1%} of women")

In [ ]:
EDU = {1: "No formal schooling", 2: "No formal schooling",
       3: "No formal schooling", 4: "No formal schooling",
       5: "Below primary", 6: "Primary", 7: "Middle", 8: "Secondary",
       10: "Higher secondary", 11: "Diploma/certificate", 12: "Graduate",
       13: "Postgraduate+"}
EDU_ORDER = list(dict.fromkeys(EDU.values()))
MAR = {1: "Never married", 2: "Married", 3: "Widowed", 4: "Divorced/separated"}
RELG = {1: "Hindu", 2: "Muslim", 3: "Christian", 4: "Sikh"}       # others pooled
SG = {1: "ST", 2: "SC", 3: "OBC", 9: "Others"}

d = pd.DataFrame({
    "lf": (w["status"] != "out of LF").astype(float),
    "age": w["age"].astype(float),
    "edu": pd.Categorical(w["gedu_lvl"].map(EDU), categories=EDU_ORDER),
    "marital": pd.Categorical(w["marst"].map(MAR),
                              categories=["Married", "Never married", "Widowed",
                                          "Divorced/separated"]),
    "urban": (w["sec"] == 2).astype(float),
    "hhsize": w["hh_size"].astype(float),
    "kids05": w["kids05"].astype(float),
    "elderly": w["elderly"].astype(float),
    "lmpce": np.log(w["hce_tot"].astype(float) / w["hh_size"].astype(float)),
    "social": pd.Categorical(w["sg"].map(SG), categories=["Others", "ST", "SC", "OBC"]),
    "religion": pd.Categorical(w["relg"].map(RELG).fillna("Other"),
                               categories=["Hindu", "Muslim", "Christian", "Sikh",
                                           "Other"]),
    "st": w["st"].astype(int),
    "mfsu": w["mfsu"].astype(int),
    "weight": w["weight"].astype(float),
})
del w
print(f"{len(d):,} women aged 25–59; missing values: {d.isna().sum().sum()}")
print(f"Weighted LFPR, women 25–59: {np.average(d['lf'], weights=d['weight']):.1%}")

## 20.3 Who participates? A first look

In [ ]:
def wmean(g, col="lf"):
    return np.average(g[col], weights=g["weight"])


by_edu = (d.groupby(["edu", "urban"], observed=True).apply(wmean)
          .unstack("urban").rename(columns={0.0: "Rural", 1.0: "Urban"}) * 100)
n_edu = d.groupby(["edu", "urban"], observed=True).size().unstack("urban")
n_edu.columns = ["n rural", "n urban"]
pd.concat([by_edu.round(1), n_edu], axis=1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))
xpos = np.arange(len(EDU_ORDER))
ax.plot(xpos, by_edu["Rural"], marker="o", color="#1f4e79", label="Rural")
ax.plot(xpos, by_edu["Urban"], marker="o", color="#c55a11", label="Urban")
ax.set_xticks(xpos)
ax.set_xticklabels(EDU_ORDER, rotation=30, ha="right")
ax.set_ylabel("LFPR, % of women aged 25–59")
ax.set_ylim(0, 80)
ax.set_title("Female participation by education: a U-shape, 2023–24",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.text(0.01, -0.14, "Usual status (ps+ss), weighted. Diploma/certificate holders "
         "are a small group with technical or vocational training.\nSource: "
         "author's calculation from PLFS 2023–24 unit-level data (NSO, MoSPI).",
         fontsize=8, color="grey")
plt.show()

## 20.4 The linear probability model

In [ ]:
RHS = ("age + I(age**2) + C(edu) + C(marital) + urban + hhsize + kids05"
       " + elderly + lmpce + C(social) + C(religion)")
CLUSTER = {"cov_type": "cluster", "cov_kwds": {"groups": d["mfsu"]}}

lpm = smf.ols("lf ~ " + RHS, data=d).fit(**CLUSTER)
fitted = lpm.fittedvalues
print(f"Predicted probabilities below 0: {(fitted < 0).mean():.1%}, "
      f"above 1: {(fitted > 1).mean():.1%}")
lpm_coef = lpm.params.drop("Intercept")
del lpm, fitted                     # keep only what we need: memory is scarce

## 20.5 Logit and probit

In [ ]:
logit = smf.logit("lf ~ " + RHS, data=d).fit(disp=0, **CLUSTER)
probit = smf.probit("lf ~ " + RHS, data=d).fit(disp=0, **CLUSTER)
print(logit.summary().tables[1])

In [ ]:
ratio = (logit.params / probit.params).drop("Intercept")
print(f"Logit/probit coefficient ratio: median {ratio.median():.2f}")
print(f"Log-likelihood: logit {logit.llf:,.0f}, probit {probit.llf:,.0f}")

## 20.6 Average marginal effects

In [ ]:
mem = logit.get_margeff(at="mean", dummy=True).summary_frame()
mem.loc[["urban", "kids05"], ["dy/dx", "Std. Err."]].round(4)

In [ ]:
def ame_by_hand(res, G, g, dg):
    """Average marginal effects and delta-method SEs for a logit or probit fit."""
    X = res.model.exog
    b = res.params.to_numpy()
    xb = X @ b
    n = X.shape[0]
    rows, grads = {}, []
    for j, name in enumerate(res.model.exog_names):
        if name == "Intercept":
            continue
        col = X[:, j]
        if np.isin(col, [0.0, 1.0]).all():                  # a dummy
            base = xb - col * b[j]
            g1, g0 = g(base + b[j]), g(base)
            rows[name] = np.mean(G(base + b[j]) - G(base))
            grad = X.T @ (g1 - g0) / n                      # d AME / d beta
            grad[j] = g1.mean()
        else:                                               # continuous
            rows[name] = np.mean(g(xb)) * b[j]
            grad = b[j] * (X.T @ dg(xb)) / n
            grad[j] += np.mean(g(xb))
        grads.append(grad)
    J = np.array(grads)
    se = np.sqrt(np.diag(J @ res.cov_params().to_numpy() @ J.T))
    return pd.DataFrame({"dy/dx": rows, "Std. Err.": se}, index=list(rows))


LOGIT_FUNS = (stats.logistic.cdf, stats.logistic.pdf,
              lambda z: stats.logistic.pdf(z) * (1 - 2 * stats.logistic.cdf(z)))
PROBIT_FUNS = (stats.norm.cdf, stats.norm.pdf, lambda z: -z * stats.norm.pdf(z))

In [ ]:
import gc

gc.collect()                           # return freed memory before a big step
small = d.sample(10_000, random_state=20)
logit_small = smf.logit("lf ~ " + RHS, data=small).fit(
    disp=0, cov_type="cluster", cov_kwds={"groups": small["mfsu"]})
check_me = logit_small.get_margeff(at="overall", dummy=True).summary_frame()
mine = ame_by_hand(logit_small, *LOGIT_FUNS)
print(f"Largest difference in AMEs: "
      f"{(mine['dy/dx'] - check_me['dy/dx']).abs().max():.1e}")
print(f"Largest difference in SEs:  "
      f"{(mine['Std. Err.'] - check_me['Std. Err.']).abs().max():.1e}")
del small, logit_small, check_me

In [ ]:
me_logit = ame_by_hand(logit, *LOGIT_FUNS)
me_probit = ame_by_hand(probit, *PROBIT_FUNS)
del probit


def tidy(name):
    """Shorten patsy names like 'C(edu)[T.Graduate]' to 'Graduate'."""
    return name.split("[T.")[1].rstrip("]") if "[T." in name else name


ame = pd.DataFrame({
    "LPM": lpm_coef,
    "logit AME": me_logit["dy/dx"],
    "probit AME": me_probit["dy/dx"],
    "logit SE": me_logit["Std. Err."],
}).rename(index=tidy)
(100 * ame.drop(["age", "I(age ** 2)"])).round(1)

In [ ]:
b = logit.params
xb = logit.fittedvalues.to_numpy()                   # linear index x'b
dens = stats.logistic.pdf(xb)                        # g(x'b) for logit
age = d["age"].to_numpy()
ame_age = np.mean(dens * (b["age"] + 2 * b["I(age ** 2)"] * age))
print(f"AME of one more year of age: {100 * ame_age:.2f} percentage points")
for a in [30, 45, 55]:
    print(f"  at age {a}: {100 * np.mean(dens * (b['age'] + 2 * b['I(age ** 2)'] * a)):.2f}")

In [ ]:
show = ["Primary", "Secondary", "Higher secondary", "Graduate", "Postgraduate+",
        "Never married", "kids05", "elderly", "urban", "lmpce", "ST", "SC",
        "Muslim", "Sikh"]
labels = {"kids05": "Each child aged 0–5", "elderly": "Each member aged 60+",
          "urban": "Urban", "lmpce": "log MPCE (+1)", "ST": "Scheduled Tribe",
          "SC": "Scheduled Caste", "Never married": "Never married (vs married)"}
pl = ame.loc[show[::-1]]

fig, ax = plt.subplots(figsize=(8, 5.6))
ypos = np.arange(len(pl))
ax.errorbar(100 * pl["logit AME"], ypos, xerr=196 * pl["logit SE"], fmt="o",
            color="#1f4e79", capsize=3, label="Logit AME (95% CI)")
ax.plot(100 * pl["LPM"], ypos, "x", color="#c55a11", label="LPM coefficient")
ax.axvline(0, color="black", lw=0.8)
ax.set_yticks(ypos)
ax.set_yticklabels([labels.get(i, i) for i in pl.index])
ax.set_xlabel("Change in probability of participation, percentage points")
ax.set_title("What goes with women's participation? (women 25–59)",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
fig.text(0.01, -0.03, "Education effects relative to no formal schooling; social "
         "group relative to 'others'; religion relative to Hindu. SEs clustered by "
         "FSU.\nSource: author's calculation from PLFS 2023–24 unit-level data "
         "(NSO, MoSPI), unweighted.", fontsize=8, color="grey")
plt.tight_layout()
plt.show()

## 20.7 Predicted probabilities across education

In [ ]:
X = pd.DataFrame(logit.model.exog, columns=logit.model.exog_names)
edu_cols = [c for c in X.columns if c.startswith("C(edu)")]
xb_no_edu = xb - X[edu_cols].to_numpy() @ b[edu_cols].to_numpy()
xb_no_edu_urban = xb_no_edu - X["urban"].to_numpy() * b["urban"]
wt = d["weight"].to_numpy()

adjusted = {}
for sector, u in [("Rural", 0.0), ("Urban", 1.0)]:
    rows = {}
    for k in EDU_ORDER:
        col = f"C(edu)[T.{k}]"
        z = xb_no_edu_urban + u * b["urban"] + (b[col] if col in b else 0.0)
        rows[k] = 100 * np.average(stats.logistic.cdf(z), weights=wt)
    adjusted[sector] = pd.Series(rows)
adjusted = pd.DataFrame(adjusted)
pd.concat({"raw": by_edu, "adjusted (logit)": adjusted}, axis=1).round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
for ax, sector, colour in zip(axes, ["Rural", "Urban"], ["#1f4e79", "#c55a11"]):
    ax.plot(xpos, by_edu[sector], marker="o", color=colour, alpha=0.4,
            label="Raw (weighted)")
    ax.plot(xpos, adjusted[sector], marker="o", color=colour,
            label="Holding other characteristics fixed")
    ax.set_xticks(xpos)
    ax.set_xticklabels(EDU_ORDER, rotation=40, ha="right", fontsize=8)
    ax.set_title(sector, loc="left", fontsize=12)
    ax.set_ylim(0, 80)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("LFPR, % of women aged 25–59")
axes[0].legend(frameon=False, fontsize=8, loc="lower left")
fig.suptitle("The U-shape survives controls", x=0.01, ha="left", fontsize=13)
fig.text(0.01, -0.12, "Adjusted: average predicted probability from the logit if "
         "every woman had the given education level and sector, weighted.\nSource: "
         "author's calculation from PLFS 2023–24 unit-level data (NSO, MoSPI).",
         fontsize=8, color="grey")
plt.tight_layout()
plt.show()

In [ ]:
rural = d[(d["urban"] == 0) & d["edu"].isin(["No formal schooling", "Graduate"])]
profile = rural.assign(
    married=(rural["marital"] == "Married").astype(float),
    tribal=(rural["social"] == "ST").astype(float),
    mpce=np.exp(rural["lmpce"])).groupby("edu", observed=True).apply(
    lambda g: pd.Series({c: np.average(g[c], weights=g["weight"])
                         for c in ["age", "married", "tribal", "kids05", "mpce"]}))
profile.round(2)

## 20.8 Adding state effects

In [ ]:
logit_st = smf.logit("lf ~ " + RHS + " + C(st)", data=d).fit(disp=0, **CLUSTER)
ame_st = ame_by_hand(logit_st, *LOGIT_FUNS)["dy/dx"]

cols = {"urban": "Urban", "C(marital)[T.Never married]": "Never married",
        "C(social)[T.ST]": "Scheduled Tribe", "C(religion)[T.Muslim]": "Muslim",
        "C(edu)[T.Graduate]": "Graduate", "C(edu)[T.Postgraduate+]": "Postgraduate+",
        "kids05": "Each child aged 0–5", "lmpce": "log MPCE (+1)"}
compare_st = pd.DataFrame({"without state effects": me_logit["dy/dx"][list(cols)],
                           "with state effects": ame_st[list(cols)]})
(100 * compare_st.rename(index=cols)).round(1)

## 20.9 Goodness of fit

In [ ]:
from sklearn.metrics import roc_auc_score

y = d["lf"].to_numpy()
fit_table = {}
for name, res in [("logit", logit), ("logit + state effects", logit_st)]:
    phat = res.predict()
    fit_table[name] = {"pseudo R²": res.prsquared,
                       "% correctly predicted": 100 * np.mean((phat > 0.5) == y),
                       "AUC": roc_auc_score(y, phat)}
fit_table["predict the majority class"] = {
    "pseudo R²": 0.0, "% correctly predicted": 100 * max(y.mean(), 1 - y.mean()),
    "AUC": 0.5}
pd.DataFrame(fit_table).T.round(3)

## 20.10 Survey weights and clustered standard errors in a logit

In [ ]:
import warnings

d["w_norm"] = d["weight"] / d["weight"].mean()
with warnings.catch_warnings():   # statsmodels warns: see the note below
    warnings.simplefilter("ignore")
    glm_w = smf.glm("lf ~ " + RHS, data=d, family=sm.families.Binomial(),
                    var_weights=d["w_norm"]).fit(**CLUSTER)
glm_f = smf.glm("lf ~ " + RHS, data=d, family=sm.families.Binomial(),
                freq_weights=d["weight"]).fit()

terms = ["urban", "kids05", "C(edu)[T.Graduate]", "C(marital)[T.Never married]"]
pd.DataFrame({
    "unweighted coef": logit.params[terms],
    "unweighted SE (cluster)": logit.bse[terms],
    "weighted coef": glm_w.params[terms],
    "weighted SE (var_weights, cluster)": glm_w.bse[terms],
    "SE with raw freq_weights": glm_f.bse[terms],
}).rename(index=tidy).round(4)

## 20.11 What these models do not tell us

In [ ]:
del d, X, logit_st, glm_f                            # free memory

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading